# **Konfiguracja Środowiska**

Ten notebook służy do **testowania** wytrenowanego modelu.

Możesz:
- Rozpoznawać emocje na pojedynczych zdjęciach
- Przetwarzać wiele zdjęć naraz
- Analizować wideo
- Rozpoznawać emocje w czasie rzeczywistym z kamery

In [1]:
# Sprawdzenie dostępności GPU
# GPU przyspiesza inference (wnioskowanie), choć nie jest tak krytyczne jak podczas treningu
%tensorflow_version 2.x
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not found')
print('Znaleziono GPU: {}'.format(device_name))

Found GPU at: /device:GPU:0


In [2]:
# Montowanie Google Drive
# Potrzebujemy dostępu do wytrenowanych wag modelu
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# Instalacja mediapipe - biblioteki Google do detekcji twarzy
# Mediapipe znajdzie twarze na zdjęciu, a nasz model rozpozna emocje
!pip install mediapipe

In [42]:
# Import wszystkich niezbędnych bibliotek

import numpy as np                    # Operacje na tablicach
import cv2                            # OpenCV - przetwarzanie obrazów i wideo
import mediapipe as mp                # Google mediapipe - detekcja twarzy
import time                           # Pomiar czasu
import glob                           # Wyszukiwanie plików
from google.colab.patches import cv2_imshow  # Wyświetlanie obrazów w Colab

# Keras/TensorFlow do budowy i ładowania modelu
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, Flatten
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adam

In [ ]:
# Kopiowanie folderu projektu z Google Drive
# UWAGA: Dostosuj ścieżkę do swojej struktury folderów!
%cp -av "/content/gdrive/MyDrive/loopQ/project" "/content"

In [6]:
# Przejście do katalogu projektu
%cd /content/project

/content/project


# **Parametry i Model**

Definiujemy:
- Mapowanie emocji na kolory (wizualizacja)
- Ścieżki do wytrenowanych wag
- Architekturę modelu VGGNet

In [7]:
# Słownik emocji z kolorami do wizualizacji
# Format: ID: [nazwa, kolor_ramki (BGR), kolor_tekstu (BGR)]
emotions = {
    0: ['Angry', (0,0,255), (255,255,255)],        # Czerwony
    1: ['Disgust', (0,102,0), (255,255,255)],      # Ciemnozielony
    2: ['Fear', (255,255,153), (0,51,51)],         # Jasny żółty
    3: ['Happy', (153,0,153), (255,255,255)],      # Fioletowy
    4: ['Sad', (255,0,0), (255,255,255)],          # Niebieski
    5: ['Surprise', (0,255,0), (255,255,255)],     # Zielony
    6: ['Neutral', (160,160,160), (255,255,255)]   # Szary
}

num_classes = len(emotions)
input_shape = (48, 48, 1)  # Rozmiar obrazu wejściowego

# Ścieżki do wytrenowanych wag
# Model 1: wytrenowany na oryginalnych danych
weights_1 = 'saved_models/vggnet.h5'
# Model 2: wytrenowany na danych po upsamplingowaniu
weights_2 = 'saved_models/vggnet_up.h5'

In [8]:
class VGGNet(Sequential):
    """
    Architektura VGGNet - identyczna jak w training.ipynb
    
    Tu tylko definiujemy strukturę - wagi zostaną wczytane z pliku.
    """
    def __init__(self, input_shape, num_classes, checkpoint_path, lr=1e-3):
        super().__init__()
        # Normalizacja
        self.add(Rescaling(1./255, input_shape=input_shape))
        
        # Blok 1: 64 filtry
        self.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal'))
        self.add(BatchNormalization())
        self.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.5))

        # Blok 2: 128 filtrów
        self.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.4))

        # Blok 3: 256 filtrów
        self.add(Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.5))

        # Blok 4: 512 filtrów
        self.add(Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.4))

        # Flatten i warstwy fully connected
        self.add(Flatten())
        self.add(Dense(1024, activation='relu'))
        self.add(Dropout(0.5))
        self.add(Dense(256, activation='relu'))

        # Warstwa wyjściowa
        self.add(Dense(num_classes, activation='softmax'))

        self.compile(optimizer=Adam(learning_rate=lr),
                    loss=categorical_crossentropy,
                    metrics=['accuracy'])
        
        self.checkpoint_path = checkpoint_path

In [9]:
# Utworzenie i wczytanie DWÓCH modeli
# Użyjemy voting (głosowania) - połączymy ich predykcje dla lepszej dokładności

# Model 1: trenowany na oryginalnych danych
model_1 = VGGNet(input_shape, num_classes, weights_1)
model_1.load_weights(model_1.checkpoint_path)
print("✓ Model 1 wczytany (oryginalny zbiór danych)")

# Model 2: trenowany na danych po upsamplingowaniu
model_2 = VGGNet(input_shape, num_classes, weights_2)
model_2.load_weights(model_2.checkpoint_path)
print("✓ Model 2 wczytany (upsampled dane)")

print("\n=== OBA MODELE GOTOWE DO UŻYCIA ===")

# **Funkcje do Inference**

**Pipeline rozpoznawania emocji:**
1. **Detekcja twarzy** - mediapipe znajduje twarze na obrazie
2. **Preprocessing** - przygotowanie wykrytych twarzy do modelu
3. **Predykcja** - model przewiduje emocje
4. **Wizualizacja** - rysowanie ramek i etykiet na obrazie

In [10]:
# Inicjalizacja detektora twarzy mediapipe
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# Utworzenie obiektu do detekcji
# min_detection_confidence=0.5 oznacza: wykryj twarz jeśli pewność >= 50%
face_detection = mp_face_detection.FaceDetection(min_detection_confidence=0.5)

print("✓ Detektor twarzy zainicjalizowany")

In [21]:
def detection_preprocessing(image, h_max=360):
    """
    Przygotowuje obraz do detekcji twarzy.
    
    Zmniejsza duże obrazy do h_max wysokości,
    co przyspiesza detekcję bez utraty dokładności.
    """
    h, w, _ = image.shape
    if h > h_max:
        ratio = h_max / h
        w_ = int(w * ratio)
        image = cv2.resize(image, (w_, h_max))
    return image

def resize_face(face):
    """
    Zmienia rozmiar pojedynczej twarzy do 48x48 pikseli.
    
    Nasz model oczekuje obrazów 48x48, więc musimy
    przeskalować wykryte twarze do tego rozmiaru.
    """
    x = tf.expand_dims(tf.convert_to_tensor(face), axis=2)
    return tf.image.resize(x, (48, 48))

def recognition_preprocessing(faces):
    """
    Przygotowuje listę twarzy do rozpoznawania emocji.
    
    Parametry:
    ----------
    faces : list
        Lista obrazów twarzy (różnych rozmiarów)
    
    Zwraca:
    -------
    x : tensor
        Batch gotowy do model.predict()
    """
    # Przeskalowanie wszystkich twarzy do 48x48
    x = tf.convert_to_tensor([resize_face(f) for f in faces])
    return x

In [27]:
def inference(image):
    """
    Główna funkcja do rozpoznawania emocji.
    
    Pipeline:
    1. Konwersja BGR -> RGB (mediapipe potrzebuje RGB)
    2. Detekcja twarzy
    3. Wydobycie i przygotowanie twarzy
    4. Predykcja emocji (voting z 2 modeli)
    5. Wizualizacja wyników
    
    Parametry:
    ----------
    image : numpy array
        Obraz BGR (format OpenCV)
    
    Zwraca:
    -------
    image : numpy array
        Obraz z narysowanymi ramkami i etykietami
    """
    H, W, _ = image.shape
    
    # Krok 1: Konwersja kolorów BGR -> RGB
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Krok 2: Detekcja twarzy
    results = face_detection.process(rgb_image)

    # Jeśli znaleziono jakieś twarze
    if results.detections:
        faces = []  # Lista wykrytych twarzy
        pos = []    # Lista pozycji (do rysowania ramek)
        
        # Dla każdej wykrytej twarzy
        for detection in results.detections:
            # Pobierz współrzędne ramki otaczającej (bounding box)
            box = detection.location_data.relative_bounding_box

            # Konwersja ze współrzędnych względnych [0,1] do pikseli
            x = int(box.xmin * W)
            y = int(box.ymin * H)
            w = int(box.width * W)
            h = int(box.height * H)

            # Upewnienie się, że współrzędne mieszczą się w obrazie
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(x + w, W)
            y2 = min(y + h, H)

            # Krok 3: Wycięcie twarzy i konwersja do skali szarości
            face = image[y1:y2, x1:x2]
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            faces.append(face)
            pos.append((x1, y1, x2, y2))
    
        # Krok 4: Przygotowanie twarzy do predykcji
        x = recognition_preprocessing(faces)

        # Predykcja z obu modeli
        y_1 = model_1.predict(x)
        y_2 = model_2.predict(x)
        
        # VOTING: Sumujemy prawdopodobieństwa i wybieramy klasę z max sumą
        # To zwiększa dokładność!
        l = np.argmax(y_1 + y_2, axis=1)

        # Krok 5: Wizualizacja - rysowanie ramek i etykiet
        for i in range(len(faces)):
            # Ramka wokół twarzy
            cv2.rectangle(image, (pos[i][0], pos[i][1]),
                          (pos[i][2], pos[i][3]), 
                          emotions[l[i]][1], 2, lineType=cv2.LINE_AA)
            
            # Wypełniony prostokąt dla tekstu (tło)
            cv2.rectangle(image, (pos[i][0], pos[i][1]-20),
                          (pos[i][2]+20, pos[i][1]), 
                          emotions[l[i]][1], -1, lineType=cv2.LINE_AA)
            
            # Tekst z nazwą emocji
            cv2.putText(image, f'{emotions[l[i]][0]}', 
                        (pos[i][0], pos[i][1]-5),
                        0, 0.6, emotions[l[i]][2], 2, lineType=cv2.LINE_AA)
    
    return image

## **Rozpoznawanie na Zdjęciach**

Przetestujmy model na pojedynczych zdjęciach lub zestawie zdjęć.

In [47]:
def infer_single_image(path):
    """
    Rozpoznaje emocje na pojedynczym zdjęciu.
    
    Parametry:
    ----------
    path : str
        Ścieżka do pliku ze zdjęciem
    """
    # Wczytanie obrazu
    image = cv2.imread(path)
    
    # Preprocessing (zmniejszenie jeśli za duży)
    image = detection_preprocessing(image)
    
    # Rozpoznanie emocji
    result = inference(image)
    
    # Zapisanie wyniku
    cv2.imwrite('run/inference/out.jpg', result)

def infer_multi_images(paths):
    """
    Rozpoznaje emocje na wielu zdjęciach.
    
    Parametry:
    ----------
    paths : list
        Lista ścieżek do plików ze zdjęciami
    """
    for i, path in enumerate(paths):
        print(f"Przetwarzanie zdjęcia {i+1}/{len(paths)}...")
        image = cv2.imread(path)
        image = detection_preprocessing(image)
        result = inference(image)
        cv2.imwrite('run/inference/out_'+str(i)+'.jpg', result)
    print("✓ Wszystkie zdjęcia przetworzone!")

In [ ]:
# Test na pojedynczym zdjęciu
# UWAGA: Zmień 'multi_1.jpg' na nazwę swojego pliku!
infer_single_image('images/multi_1.jpg')

# Wyświetlenie wyniku
out = cv2.imread('run/inference/out.jpg')
cv2_imshow(out)

print("\nSprawdź:")
print("- Czy wszystkie twarze zostały wykryte?")
print("- Czy emocje są poprawnie rozpoznane?")
print("- Jakie kolory odpowiadają jakim emocjom?")

In [ ]:
# Test na wielu zdjęciach
# Wyszukanie wszystkich plików .jpg w folderze images/
paths = np.sort(np.array(glob.glob('images/*.jpg')))
print(f"Znaleziono {len(paths)} zdjęć do przetworzenia")

# Przetworzenie wszystkich
infer_multi_images(paths)

# Wyświetlenie wszystkich wyników
out_paths = np.sort(np.array(glob.glob('run/inference/*.jpg')))
for path in out_paths:
    image = cv2.imread(path)
    cv2_imshow(image)

In [ ]:
# Kopiowanie wyników do Google Drive
# UWAGA: Dostosuj ścieżkę!
%cp -av /content/project/run/inference/ /content/gdrive/MyDrive/loopQ/project/run/

## **Rozpoznawanie na Wideo**

Przetwarzanie wideo ramka po ramce.

**Jak to działa:**
1. Wczytanie wideo
2. Dla każdej ramki:
   - Wykryj twarze
   - Rozpoznaj emocje
   - Narysuj ramki i etykiety
3. Zapisanie przetworzonego wideo

In [22]:
# Konfiguracja wideo
video = 'test_video/emotions.mp4'  # Ścieżka do pliku wideo

# Otwarcie pliku wideo
cap = cv2.VideoCapture(video)

# Pobranie parametrów wideo
frame_width = int(cap.get(3))   # Szerokość
frame_height = int(cap.get(4))  # Wysokość
fps = cap.get(cv2.CAP_PROP_FPS) # Klatki na sekundę

# Zmniejszenie rozmiaru wyjściowego (przyspiesza przetwarzanie)
target_h = 360
target_w = int(target_h * frame_width / frame_height)

# Utworzenie obiektu do zapisu wideo
out = cv2.VideoWriter('run/out.avi',
                      cv2.VideoWriter_fourcc('M','J','P','G'),
                      fps, (target_w, target_h))

print(f"Wideo: {frame_width}x{frame_height} @ {fps} FPS")
print(f"Wyjście: {target_w}x{target_h}")

In [23]:
# Pętla przetwarzania wideo
print("Rozpoczęcie przetwarzania wideo...")
print("To może zająć kilka minut...\n")

frame_count = 0
while True:
    # Odczyt następnej ramki
    success, image = cap.read()
    
    if success:
        frame_count += 1
        if frame_count % 30 == 0:  # Co 30 ramek wyświetl postęp
            print(f"Przetworzone ramki: {frame_count}")
        
        # Zmniejszenie rozmiaru ramki
        image = detection_preprocessing(image)
        
        # Rozpoznanie emocji na ramce
        result = inference(image)
        
        # Zapisanie przetworzonej ramki
        out.write(result)
        
        # Opcjonalne: przerwanie przez naciśnięcie 'q' (w lokalnym środowisku)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    else:
        # Koniec wideo
        break
    
# Zamknięcie plików
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"\n✓ Przetwarzanie zakończone!")
print(f"Przetworzone ramki: {frame_count}")
print(f"Wideo zapisane w: run/out.avi")

# **Rozpoznawanie z Kamery (Google Colab)**

W Google Colab możemy użyć kamery przeglądarki do robienia zdjęć
i rozpoznawania emocji w czasie rzeczywistym!

**Uwaga:** To działa tylko w Google Colab, nie w lokalnym Jupyter.

In [52]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='cap/photo.jpg', quality=0.8):
    """
    Uruchamia kamerę w przeglądarce i robi zdjęcie.
    
    Jak użyć:
    1. Uruchom komórkę
    2. Zezwól przeglądarce na dostęp do kamery
    3. Kliknij "Capture" gdy jesteś gotowy
    4. Zdjęcie zostanie zapisane i przetworzone
    """
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

In [ ]:
# Zrób zdjęcie i rozpoznaj emocje!
print("Kliknij 'Capture' gdy będziesz gotowy...")
filename = take_photo()

print("\nPrzetwarzanie zdjęcia...")
image = cv2.imread(filename)
result = inference(image)

print("\nWynik:")
cv2_imshow(result)

print("\nJaka emocja została wykryta?")
print("Czy się zgadzasz z przewidywaniem modelu?")

# **Podsumowanie**

Gratulacje! Przetestowałeś model rozpoznawania emocji na:
- ✓ Pojedynczych zdjęciach
- ✓ Wielu zdjęciach
- ✓ Wideo
- ✓ Zdjęciach z kamery

## Obserwacje

**Co zadziałało dobrze:**
- Które emocje model rozpoznaje najlepiej?
- W jakich warunkach (oświetlenie, kąt) działa najlepiej?

**Co można poprawić:**
- Które emocje model myli?
- Czy są fałszywe detekcje (twarze gdzie ich nie ma)?
- Czy są pominięte twarze (nie wykryte)?

## Dalsze Eksperymenty

1. **Zbierz własne dane** - zrób zdjęcia z różnymi emocjami
2. **Fine-tuning** - dotrenuuj model na swoich danych
3. **Nowe architektury** - wypróbuj ResNet, EfficientNet
4. **Aplikacja** - stwórz aplikację webową z tym modelem

## Zasoby do nauki

- [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
- [Mediapipe Guide](https://google.github.io/mediapipe/)
- [OpenCV Tutorials](https://docs.opencv.org/master/d9/df8/tutorial_root.html)

**Powodzenia w dalszej nauce!** 🚀